# 🚂 RailGuard AI: Adverse Condition Training Master Plan

This notebook implements the **Two-Stage Training Strategy** (30 epochs Clean -> 70 epochs Adverse) to ensure 90%+ reliability in night, fog, and rain for **Cyber Dome 2026**.

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics roboflow opencv-python-headless
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

## 2. Dataset Preparation
Make sure you have uploaded `railfod23` and `adverse_augmented` datasets to your Drive.

In [ ]:
import os
# Symlink drive to local for faster access
!ln -s /content/drive/MyDrive/railguard_datasets /content/datasets

## 3. STAGE 1: Clean Domain Training (30 Epochs)
Establish baseline accuracy on high-quality daylight/standard imagery.

In [ ]:
model = YOLO('yolov8s.pt') # Start with Small for balance

results = model.train(
    data='/content/datasets/railfod23/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    freeze=9,           # Freeze backbone for initial stability
    lr0=0.001,
    name='stage1_clean',
    device=0, 
    half=True,
    project='/content/drive/MyDrive/railguard/runs'
)

## 4. STAGE 2: Adverse Domain Fine-Tuning (70 Epochs)
Focus on Night, Fog, and Rain using **Extreme Augmentation** (Mosaic, Mixup, HSV-V).

In [ ]:
# Load the best weights from Stage 1
model2 = YOLO('/content/drive/MyDrive/railguard/runs/stage1_clean/weights/best.pt')

results_adverse = model2.train(
    data='/content/datasets/adverse_augmented/data.yaml',
    epochs=70,
    imgsz=640,
    batch=16,
    freeze=0,           # Unfreeze everything for full domain adaptation
    
    # --- EXTREME AUGMENTATION (Research-Backed) ---
    hsv_v=0.6,          # High brightness variance (Night/Sodium Vapor)
    mosaic=1.0,         # 4-image tiling
    mixup=0.2,          # Alpha-blending (Fog simulation)
    erasing=0.5,        # Occlusion handling (Rain streaks/Debris)
    
    lr0=0.0001,         # Lower Base LR for fine-tuning
    lrf=0.01,
    cos_lr=True,
    name='stage2_adverse_final',
    device=0, 
    half=True,
    patience=20,
    save=True,
    project='/content/drive/MyDrive/railguard/runs'
)

## 5. Export & Victory Lap

In [ ]:
!cp /content/drive/MyDrive/railguard/runs/stage2_adverse_final/weights/best.pt /content/drive/MyDrive/RailGuard_Adverse_GoldMaster.pt
print("Training COMPLETED. Final Gold Master weights saved to Drive.")